<a href="https://colab.research.google.com/github/Mahammad-Haroon/Data-Analysis-Projects/blob/main/Schema_Design_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import SQLite

In [1]:
import sqlite3
import pandas as pd

# Create an in-memory database
conn = sqlite3.connect(":memory:")

print("Database connected successfully!")

Database connected successfully!


#Create the Tables

In [2]:
# Create DimCustomer
conn.execute("""
CREATE TABLE DimCustomer (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    segment TEXT,
    city TEXT,
    region TEXT,
    age_group TEXT,
    acquisition_channel TEXT
);
""")

# Create DimProduct
conn.execute("""
CREATE TABLE DimProduct (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    sub_category TEXT,
    brand TEXT,
    price_tier TEXT
);
""")

# Create DimDate
conn.execute("""
CREATE TABLE DimDate (
    date_id INTEGER PRIMARY KEY,
    full_date TEXT,
    day_of_week TEXT,
    month INTEGER,
    quarter TEXT,
    financial_year INTEGER,
    is_weekend INTEGER
);
""")

# Create FactSales
conn.execute("""
CREATE TABLE FactSales (
    sale_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    date_id INTEGER,
    sale_amount REAL,
    quantity_sold INTEGER,
    discount_amount REAL,

    FOREIGN KEY (customer_id)
        REFERENCES DimCustomer(customer_id),

    FOREIGN KEY (product_id)
        REFERENCES DimProduct(product_id),

    FOREIGN KEY (date_id)
        REFERENCES DimDate(date_id)
);
""")

print("All four tables created successfully!")

All four tables created successfully!


#Enable Foreign Key Checking

In [3]:
conn.execute("PRAGMA foreign_keys = ON;")

print("Foreign key constraints enabled.")

Foreign key constraints enabled.


#Task 1: Insert DimCustomer

In [4]:
conn.execute("""
INSERT INTO DimCustomer
(
    customer_id,
    customer_name,
    segment,
    city,
    region,
    age_group,
    acquisition_channel
)
VALUES
(
    106,
    'Pooja Patel',
    'Premium',
    'Bangalore',
    'South',
    '25-34',
    'Online'
);
""")

conn.commit()

print("Task 1 completed: Customer inserted.")

Task 1 completed: Customer inserted.


#Task 2: Insert DimProduct

In [5]:
conn.execute("""
INSERT INTO DimProduct
(
    product_id,
    product_name,
    category,
    sub_category,
    brand,
    price_tier
)
VALUES
(
    206,
    'Wireless Earbuds',
    'Electronics',
    'Audio',
    'JBL',
    'Mid'
);
""")

conn.commit()

print("Task 2 completed: Product inserted.")

Task 2 completed: Product inserted.


#Task 3: Insert DimDate

In [6]:
conn.execute("""
INSERT INTO DimDate
(
    date_id,
    full_date,
    day_of_week,
    month,
    quarter,
    financial_year,
    is_weekend
)
VALUES
(
    306,
    '2024-09-28',
    'Saturday',
    9,
    'Q3',
    2024,
    1
);
""")

conn.commit()

print("Task 3 completed: Date inserted.")

Task 3 completed: Date inserted.


#Task 4: Insert FactSales

In [7]:
conn.execute("""
INSERT INTO FactSales
(
    sale_id,
    customer_id,
    product_id,
    date_id,
    sale_amount,
    quantity_sold,
    discount_amount
)
VALUES
(
    1006,
    106,
    206,
    306,
    4500.00,
    1,
    500.00
);
""")

conn.commit()

print("Task 4 completed: Sales record inserted.")

Task 4 completed: Sales record inserted.


#Verify DimCustomer

In [8]:
pd.read_sql_query("""
SELECT *
FROM DimCustomer
WHERE customer_id = 106;
""", conn)

,customer_id,customer_name,segment,city,region,age_group,acquisition_channel
0,106,Pooja Patel,Premium,Bangalore,South,25-34,Online


#Verify DimProduct

In [9]:
pd.read_sql_query("""
SELECT *
FROM DimProduct
WHERE product_id = 206;
""", conn)

,product_id,product_name,category,sub_category,brand,price_tier
0,206,Wireless Earbuds,Electronics,Audio,JBL,Mid


#Verify DimDate

In [10]:
pd.read_sql_query("""
SELECT *
FROM DimDate
WHERE date_id = 306;
""", conn)

,date_id,full_date,day_of_week,month,quarter,financial_year,is_weekend
0,306,2024-09-28,Saturday,9,Q3,2024,1


#Verify FactSales

In [11]:
pd.read_sql_query("""
SELECT *
FROM FactSales
WHERE sale_id = 1006;
""", conn)

,sale_id,customer_id,product_id,date_id,sale_amount,quantity_sold,discount_amount
0,1006,106,206,306,4500.0,1,500.0


#Run the JOIN Query

In [12]:
query = """
SELECT
    c.region,
    p.category,
    d.quarter,
    SUM(f.sale_amount) AS total_sales,
    SUM(f.quantity_sold) AS total_quantity,
    SUM(f.discount_amount) AS total_discount
FROM FactSales f
JOIN DimCustomer c
    ON f.customer_id = c.customer_id
JOIN DimProduct p
    ON f.product_id = p.product_id
JOIN DimDate d
    ON f.date_id = d.date_id
GROUP BY
    c.region,
    p.category,
    d.quarter
ORDER BY
    c.region,
    p.category;
"""

print(query)


SELECT
    c.region,
    p.category,
    d.quarter,
    SUM(f.sale_amount) AS total_sales,
    SUM(f.quantity_sold) AS total_quantity,
    SUM(f.discount_amount) AS total_discount
FROM FactSales f
JOIN DimCustomer c
    ON f.customer_id = c.customer_id
JOIN DimProduct p
    ON f.product_id = p.product_id
JOIN DimDate d
    ON f.date_id = d.date_id
GROUP BY
    c.region,
    p.category,
    d.quarter
ORDER BY
    c.region,
    p.category;



#Execute the Query

In [13]:
result = pd.read_sql_query(query, conn)

result

,region,category,quarter,total_sales,total_quantity,total_discount
0,South,Electronics,Q3,4500.0,1,500.0


#Final Verification
To demonstrate that all four records are connected correctly, run this:

In [14]:
pd.read_sql_query("""
SELECT
    f.sale_id,
    c.customer_name,
    c.city,
    c.region,
    p.product_name,
    p.category,
    p.brand,
    d.full_date,
    d.quarter,
    f.sale_amount,
    f.quantity_sold,
    f.discount_amount
FROM FactSales f
JOIN DimCustomer c
    ON f.customer_id = c.customer_id
JOIN DimProduct p
    ON f.product_id = p.product_id
JOIN DimDate d
    ON f.date_id = d.date_id
WHERE f.sale_id = 1006;
""", conn)

,sale_id,customer_name,city,region,product_name,category,brand,full_date,quarter,sale_amount,quantity_sold,discount_amount
0,1006,Pooja Patel,Bangalore,South,Wireless Earbuds,Electronics,JBL,2024-09-28,Q3,4500.0,1,500.0
